# Data Description
- Background: 2026 縣市長登記名冊、公報發布前的候選人資料準備，以及舊版縣市長參選紀錄。資料用於既有提案的候選人分群與策略比較。
- date received: 20261017（PDF 本次下載；歷史資料本次自既有 election 資料包複製）
- Path to data file: [中選會登記名冊](https://web.cec.gov.tw/central/article/64709)；歷史來源為 `election/processed/candidate_election_results.csv`。
- Unit of observation: 登記表每列是一位候選人的當次登記；歷史表每列是一位候選人的一次縣市長參選。
- Sample period: 登記表製表日 2026-09-07，歷史表 2001–2022（含資料包內補選）。
- Known issues: 登記名冊不等於最終審定名單；尚須對照資格審查與後續公告。歷史資料僅涵蓋縣市長，不能證明首次參選任何公職，也未涵蓋 2001 年以前。姓名比對只能產生人工核對線索。歷史學歷不可當成 2026 學歷。原始 PDF 分頁換行姓名只在轉檔時合併，原始檔不動。
- Definition for each variable: source table 的 county_city 為選舉區、registration_date_roc 為民國登記日期、candidate_name 為姓名、party_recommendation_original 為推薦政黨原文、source_note 為原備註、office_type 為 C1 直轄市長／C2 縣市長、source_file/page/url 為來源定位。歷史表欄位沿用 [既有說明](../../../election/README_legacy.md)，其產生程式非本次撰寫。
- Data path: 下方逐檔列示，皆為 `data/raw/`。
- Role: raw／既有外部整理表的固定快照。
- Verification status: 已執行 SHA-256、讀取、樣本與列數檢查；不表示完成資格或身分核實。
- SHA-256: 下列各檔雜湊。
- Next processing notebook: [候選人資料準備](../code/prepare_candidates_2026.ipynb)。PDF 轉表程式：[extract_registration_pdfs.py](../code/extract_registration_pdfs.py)，需 pdfplumber；轉表 CSV 是本次衍生的來源轉檔，不冒稱官方發布 CSV。

- `2026_metropolitan_registration.pdf`: `5ab6ec29145feab84f8e697495f039217bb18ace252382e7e150bb1ea7ae18fa`
- `2026_county_registration.pdf`: `97c32aa206bfd5d1e07d5b9658f853b6b038053b662dd04fdd0132af614a8136`
- `cec_registration_2026_source_table.csv`: `c057759cabef3f0b6f7a9137d8e85c9afee2b45358c8c31e764ced4cae8c85f0`
- `historical_mayor_candidates_2001_2022.csv`: `ed8f63163d5f0c82301671301e658167cdc4eea5fa81aee8d9078c3226abb097`

In [1]:
from pathlib import Path
import hashlib
import pandas as pd


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "AGENTS.md").is_file() and (candidate / "templates").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the student project containing AGENTS.md and templates/.")


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    # Explicit project-relative paths only; never select the first matching file.
    rel = Path(relative_path)
    if rel.is_absolute() or ".." in rel.parts or "<" in str(rel):
        raise ValueError(f"Replace the placeholder with an exact project-relative path: {relative_path}")
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / "data" / area).resolve()
    if allowed == PROJECT_ROOT or not allowed.is_relative_to(PROJECT_ROOT):
        raise ValueError(f"data/{area}/ resolves outside the project root.")
    if not path.is_relative_to(allowed):
        raise ValueError(f"Expected a file inside data/{area}/: {relative_path}")
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def verify_input_sha256(path, expected):
    # Bind this run to the exact file inspected when the notebook was prepared.
    if (not isinstance(expected, str) or len(expected) != 64
            or any(char not in "0123456789abcdefABCDEF" for char in expected)):
        raise ValueError("Fill the expected SHA-256 from the inspected input before loading data.")
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f"Input changed: identify the new version before loading {path.name}.")
    return actual

EXPECTED = {'2026_metropolitan_registration.pdf': '5ab6ec29145feab84f8e697495f039217bb18ace252382e7e150bb1ea7ae18fa', '2026_county_registration.pdf': '97c32aa206bfd5d1e07d5b9658f853b6b038053b662dd04fdd0132af614a8136', 'cec_registration_2026_source_table.csv': 'c057759cabef3f0b6f7a9137d8e85c9afee2b45358c8c31e764ced4cae8c85f0', 'historical_mayor_candidates_2001_2022.csv': 'ed8f63163d5f0c82301671301e658167cdc4eea5fa81aee8d9078c3226abb097'}
for name, fingerprint in EXPECTED.items():
    verify_input_sha256(project_file("data/raw/"+name,"raw"), fingerprint)
print("All four source fingerprints verified")

All four source fingerprints verified


## Change for Your Own Data
固定來源路徑與雜湊，資料更新時先核對版本。

In [2]:
source = pd.read_csv(project_file("data/raw/cec_registration_2026_source_table.csv","raw"),dtype=str,keep_default_na=False)
history = pd.read_csv(project_file("data/raw/historical_mayor_candidates_2001_2022.csv","raw"),dtype=str,keep_default_na=False)

## Summary

In [3]:
print({"registration_rows":len(source),"counties":source.county_city.nunique(),"historical_candidate_rows":len(history)})
assert len(source)==81 and source.county_city.nunique()==22
assert len(history)==538

{'registration_rows': 81, 'counties': 22, 'historical_candidate_rows': 538}


### Sample

In [4]:
display(source.head(10)); display(history[["election_date","county_city","candidate_name","sex","birth_year","education_level_cec"]].head(10))

,county_city,registration_date_roc,candidate_name,party_recommendation_original,source_note,office_type,source_file,source_page,source_url
0,臺北市,115/08/31,郭璽,台灣麻將最大黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...
1,臺北市,115/09/02,沈伯洋,民主進步黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...
2,臺北市,115/09/02,蕭文乾,臺灣SoR無法黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...
3,臺北市,115/09/03,唐新民,三勢團結促進聯盟,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...
4,臺北市,115/09/04,蔣萬安,中國國民黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...
5,臺北市,115/09/04,林志成,無,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...
6,新北市,115/09/02,蘇巧慧,民主進步黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...
7,新北市,115/09/04,李四川,中國國民黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...
8,新北市,115/09/04,蘇輝湟,無,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...
9,桃園市,115/09/01,張善政,中國國民黨,,C1,data/raw/2026_metropolitan_registration.pdf,1,https://web.cec.gov.tw/api/file/bb9a8d7a-9b8a-...


,election_date,county_city,candidate_name,sex,birth_year,education_level_cec
0,2001-12-01,南投縣,姜君佩,男,1955,
1,2001-12-01,南投縣,張明雄,男,1944,
2,2001-12-01,南投縣,陳胘富,男,1960,
3,2001-12-01,南投縣,林宗男,男,1942,
4,2001-12-01,南投縣,林明溱,男,1951,
5,2001-12-01,南投縣,陳振盛,男,1950,
6,2001-12-01,南投縣,彭百顯,男,1949,
7,2001-12-01,嘉義市,蕭裕珍,女,1956,
8,2001-12-01,嘉義市,黃正男,男,1941,
9,2001-12-01,嘉義市,張榮藏,男,1942,


### Summary Stats

In [5]:
display(source.groupby(["office_type","party_recommendation_original"]).size().rename("registrants").to_frame()); display((source=="").sum().rename("blank_cells").to_frame())

registrants
office_type party_recommendation_original             
C1          三勢團結促進聯盟                                 1
            中國國民黨                                    6
            台灣麻將最大黨                                  1
            司法改革黨                                    1
            司法正義黨                                    1
            民主進步黨                                    6
            無                                        5
            臺灣SoR無法黨                                 2
C2          中國國民黨                                   13
            台灣工黨                                     1
            台灣民眾黨                                    1
            台聯黨                                      1
            民主進步黨                                   13
            無                                       29

,blank_cells
county_city,0
registration_date_roc,0
candidate_name,0
party_recommendation_original,0
source_note,81
office_type,0
source_file,0
source_page,0
source_url,0


## 補充來源（2026-10-17 第二輪蒐集）

遠見媒體彙整檔逐一核對81人姓名、縣市、登記推薦政黨（明列縮寫與温／溫異體字），吻合不代表逐項學位已查證。空陣列與「尚未公布」仍為缺值。遠見原文學歷保存於observations之education_gvm_compilation；僅在原CNA學歷空白時補入education_media_supplement並明標學位待核實。新增羅佩秦學歷說明，原視報導可支持陸官正期53期畢業證書的說法，但未獨立確認其兩項學士學位；不改education_2026或教育分類。出生年無新增，仍缺羅佩秦、魏造文。GVM政見為媒體編輯摘要，不合併到候選人官網原文段落。

本次再查原缺性別的25人，新增19筆可追溯觀察（政府／議會／政黨／專業學會8人、媒體11人），主表80/81（新增5人依使用者提供，未獨立查證）；其中葉人文依媒體「大港男兒」明示描述，並非戶籍性別欄。原缺6人中，使用者補充王肇民、洪方隆、朱定瑀、羅佩秦、李吳穎智為男，以user_provided標記；魏造文回覆未知，主欄仍保留空白，sex_evidence_status記錄user_reported_unknown。社群目錄暫存值保留原來源。李文良僅取得官方名冊搜尋索引片段，林國漳採web工具抽出PDF文字（直接下載403），限制分列於sex_evidence_status。性別欄為公開來源標記，不代表已查驗2026戶籍資料。逐人來源與未解缺口見data/raw/sex_verification/sex_search_audit_25.csv。

- Background: 補足候選人背景與競選材料；原四份來源仍為第一版。
- date received: 20261017；原 CEC ZIP 本已保存，本次抽出其他公職候選人表。
- Path to data file / Data path / SHA-256: 固定 manifest 逐檔列示。
- Unit of observation: CEC 每列一筆公職候選人紀錄；HTML 一份來源頁面；查詢 JSON 每筆一次候選人查詢。
- Sample period: CEC 檔案涵蓋 1994–2024 的已標年份資料（並非完整終生普查），網站為 2026-10-17 擷取、底層資料日期可能較早；2024 不分區公報為 2024 選舉。
- Known issues: 原始全公職紀錄有同名不同人、空值、前置單引號與跨年學歷差異。不分區 elcand 可只列政黨，故另取 PDF。部分網站拒絕下載，保存失敗狀態。中選會資格頁 HTML 僅為動態頁框，81 位審定名單採用保存的官方頁搜尋索引片段，明列此限制。CNA 有不完整／錯字日期，保留不自動修補。HTML 分享按鈕不是候選人粉專。
- Definition for each variable: manifest 是 exact relative path → SHA-256；全公職 CSV 原始欄序見整理程式，未在 raw 階段修正；source URL / capture date / status 見各 capture manifest。
- Role: raw snapshots／忠實轉檔；不是分析輸入。
- Next processing notebook: [enrich_candidates_2026.ipynb](../code/enrich_candidates_2026.ipynb)。


In [6]:
import json
verify_input_sha256(project_file('data/raw/enrichment_input_manifest.json','raw'),'ef903cf53e78ce5615b922bdf42193b8a57d92dd84f45b4ee6d4b5127109709c')
snapshot_inputs=json.loads(project_file('data/raw/enrichment_input_manifest.json','raw').read_text())
for relative,h in snapshot_inputs.items():
    area=Path(relative).parts[1]
    verify_input_sha256(project_file(relative,area),h)
print(len(snapshot_inputs),'pinned supplementary input files verified')
display(pd.DataFrame([{'path':p,'sha256':h} for p,h in snapshot_inputs.items()]).head(10))


255 pinned supplementary input files verified


,path,sha256
0,data/processed/candidates_2026_registration.csv,765797d09de8e70f6cb54486cf660c44f93e7aab1c68c3...
1,data/raw/cna_2026_candidates_supplement.html,87fbe8057eaebcdcb48e791175f9279bceae947f600cba...
2,data/raw/ly_profile_capture_manifest.json,71ded04008946f375bc4b65ab94992569153e4fe9afec1...
3,data/raw/candidate_website_capture_manifest.json,7563a16ec0e68a055f723a30c835fe5513607c96fbc3f6...
4,data/raw/extra_campaign_capture_manifest.json,845b16de28d6cf61d042147a5eef134b1c811c7b24dab9...
5,data/raw/cec_all_offices_manifest.json,726d464020ff8f850ce39685bb70926c8529712f75c183...
6,data/raw/cec_2024_party_list_bulletin.pdf,f2203c02e758fec090b1119432e78bffb75ccef936ddf9...
7,data/raw/cec_2024_party_list_bulletin.pages.json,ff80083e62ac5d4f4bd82178db224cf985faddfd0d6f12...
8,data/raw/cec_2026_qualification_review.html,869f2002f7d0d3335a108931f10ec79a0580fa4c5a1870...
9,data/raw/facebook_access_checks_20261017.txt,1fe6fc640b00e40183201a0fa8ace3b9ca439d514ac718...
